In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.chdir(r"E:\ML_edu\adidas")

df = pd.read_csv("Adidas_Global.csv", low_memory=False)
df_products = df.drop_duplicates(subset=['country_code', 'model_number']).copy()

print(f"df (SKU):       {len(df):,} строк")
print(f"df_products:    {len(df_products):,} строк")
print(f"Колонок:        {df.shape[1]}")

df (SKU):       44,888 строк
df_products:    1,990 строк
Колонок:        35


## Урок 2. Очистка данных — начало

### Шаг 1: Загрузка и воспроизводимость
```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.chdir(r"E:\ML_edu\adidas")

df = pd.read_csv("Adidas_Global.csv", low_memory=False)
df_products = df.drop_duplicates(subset=['country_code', 'model_number']).copy()

print(f"df (SKU):       {len(df):,} строк")
print(f"df_products:    {len(df_products):,} строк")
print(f"Колонок:        {df.shape[1]}")
```

**Результат:**
```
df (SKU):       44,888 строк
df_products:    1,990 строк
Колонок:        35
```

Каждый новый блокнот начинается с полного воспроизведения стартового состояния.
Это называется **воспроизводимость** (reproducibility) — важнейший принцип в data science.
Если ты откроешь этот блокнот через месяц или отправишь коллеге — он должен
получить те же числа запустив ячейки сверху вниз.

`os.chdir()` фиксирует рабочую директорию, поэтому `"Adidas_Global.csv"` находится
без указания полного пути. Это удобнее, чем прописывать `E:\ML_edu\adidas\...` в каждой строке.

Два датафрейма, с которыми работаем в этом уроке:
- `df` — 44,888 строк, уровень размеров. Используем для анализа наличия.
- `df_products` — 1,990 строк, уровень товаров. **Всё остальное — только здесь.**

In [2]:
# Сколько строк имеют rating_count = -99?
mask_99 = df_products['rating_count'] == -99
print(f"Строк с rating_count = -99: {mask_99.sum()}")
print(f"Это {mask_99.sum() / len(df_products) * 100:.1f}% товаров")

# Посмотрим на эти строки
df_products[mask_99][['country_code', 'product_name', 'rating', 'rating_count']].head(8)

Строк с rating_count = -99: 638
Это 32.1% товаров


,country_code,product_name,rating,rating_count
0,BR,AGASALHO COM CAPUZ BIG LOGO,NaN,-99.0
134,BR,AGASALHO FIREBIRD,NaN,-99.0
142,BR,AGASALHO FIREBIRD,NaN,-99.0
147,BR,AGASALHO MOLETINHO TRÊS LISTRAS,NaN,-99.0
173,BR,AGASALHO SST,NaN,-99.0
181,BR,AGASALHO SST,NaN,-99.0
440,BR,Agasalho Yoga,NaN,-99.0
534,BR,BLUSA CAPUZ FRONTAL SEASONALS ANIMAL FECHO ZÍPER,NaN,-99.0


## Шаг 2: Аномалия rating_count = -99
```python
mask_99 = df_products['rating_count'] == -99
print(f"Строк с rating_count = -99: {mask_99.sum()}")
print(f"Это {mask_99.sum() / len(df_products) * 100:.1f}% товаров")

df_products[mask_99][['country_code', 'product_name', 'rating', 'rating_count']].head(8)
```

**Результат:** 638 строк (32.1% товаров), все из Бразилии (BR), у всех `rating = NaN`.

---

### Что мы видим

**Паттерн чёткий:** `rating_count = -99` всегда идёт вместе с `rating = NaN`.
Это значит `-99` — это не реальное число, а **технический маркер** «нет данных»,
который попал в числовую колонку вместо `NaN`.

Это очень распространённая проблема в реальных данных: системы иногда пишут
заглушки (-1, -99, 999, 0) вместо настоящего отсутствия значения.
pandas не знает об этой договорённости — для него -99 такое же число, как 42.

### Стратегия исправления

Заменяем `-99` на `NaN` — это честное представление «нет данных»:
```python
# Заменяем -99 на NaN — это честное "нет данных"
df_products['rating_count'] = df_products['rating_count'].replace(-99, np.nan)

# Проверяем что исправили
print("После исправления:")
print(f"  Строк с -99: {(df_products['rating_count'] == -99).sum()}")
print(f"  Строк с NaN: {df_products['rating_count'].isnull().sum()}")
print(f"  Min значение: {df_products['rating_count'].min()}")
```

**Важный нюанс:** `np.nan` — это специальный объект из numpy,
который pandas понимает как «пустое значение». Нельзя написать просто `None`
для числовых колонок — нужно именно `np.nan`.

In [5]:
# Заменяем -99 на NaN — это честное "нет данных"
df_products['rating_count'] = df_products['rating_count'].replace(-99, np.nan)
# Проверяем что исправили
print("После исправления:")
print(f"  Строк с -99: {(df_products['rating_count'] == -99).sum()}")
print(f"  Строк с NaN: {df_products['rating_count'].isnull().sum()}")
print(f"  Min значение: {df_products['rating_count'].min()}")

После исправления:
  Строк с -99: 0
  Строк с NaN: 1146
  Min значение: 1.0


## Шаг 3: Исправление rating_count = -99 → NaN
```python
df_products['rating_count'] = df_products['rating_count'].replace(-99, np.nan)

print("После исправления:")
print(f"  Строк с -99: {(df_products['rating_count'] == -99).sum()}")
print(f"  Строк с NaN: {df_products['rating_count'].isnull().sum()}")
print(f"  Min значение: {df_products['rating_count'].min()}")
```

**Результат:**
```
Строк с -99:  0
Строк с NaN:  1146
Min значение: 1.0
```

Аномалия устранена. Разбираем числа:

- `-99` было у 638 товаров, стало `NaN` — они добавились к уже существующим пропускам
- Итого `NaN` стало 1146 — это честная картина «у этих товаров нет отзывов»
- `min = 1.0` — теперь минимальное значение настоящее: товар с одним отзывом

Это называется **data cleaning pattern**: найти маркер → заменить на NaN → проверить.
Три строки кода, но данные стали честнее.

In [6]:
# Отрицательные скидки — что это?
neg_disc = df_products[df_products['discount_pct'] < 0]
print(f"Товаров с отрицательной скидкой: {len(neg_disc)}")
print()
neg_disc[['country_code', 'product_name', 'price_local',
          'sale_price_local', 'discount_pct']].head(10)

Товаров с отрицательной скидкой: 18



,country_code,product_name,price_local,sale_price_local,discount_pct
8950,DE,Terrex Agravic Speed Ultra Trailrunning-Schuh,149.5,161.0,-30.0
31802,KR,SAMBA OG 신발,118300.0,118300.0,-30.0
31859,KR,SL 72 OG,90300.0,90300.0,-30.0
31944,KR,Samba OG 신발,104300.0,104300.0,-30.0
31975,KR,가젤 인도어,111300.0,85000.0,-45.0
32012,KR,도쿄,97300.0,97300.0,-30.0
32031,KR,라이트블레이즈,64500.0,103200.0,-20.0
32285,KR,삼바 OG 슈즈,118300.0,118300.0,-30.0
32801,KR,아디다스 오리지널스 슈퍼스타 부츠,160300.0,183200.0,-20.0
32836,KR,아디뮬 슬라이드,129000.0,103200.0,-20.0


## Шаг 4: Аномалия — отрицательные скидки
```python
neg_disc = df_products[df_products['discount_pct'] < 0]
print(f"Товаров с отрицательной скидкой: {len(neg_disc)}")
neg_disc[['country_code', 'product_name', 'price_local',
          'sale_price_local', 'discount_pct']].head(10)
```

**Результат:** 18 товаров, почти все из Кореи (KR), один из Германии (DE).

---

### Что на самом деле происходит

Смотрим на конкретные строки и видим два разных паттерна:

**Паттерн 1 — sale_price > price (строки KR 32031, 32801, 32836):**
```
라이트블레이즈  price=64,500  sale_price=103,200  discount=-20
```
Цена "со скидкой" выше обычной цены. Это нелогично — скорее всего
перепутаны местами колонки при выгрузке данных из корейского сайта.

**Паттерн 2 — price == sale_price, но discount отрицательный (строки KR 31802, 31859...):**
```
SAMBA OG  price=118,300  sale_price=118,300  discount=-30
```
Цены одинаковые, но скидка -30. Явная ошибка в исходных данных.

### Стратегия

Отрицательная скидка экономически не имеет смысла.
Все отрицательные значения заменяем на `NaN` — «нет скидки»:
```python
# Заменяем отрицательные скидки на NaN
df_products['discount_pct'] = df_products['discount_pct'].where(
    df_products['discount_pct'] >= 0, np.nan
)

# Проверяем
print(f"Отрицательных скидок осталось: {(df_products['discount_pct'] < 0).sum()}")
print(f"Min discount_pct теперь: {df_products['discount_pct'].min()}")
print(f"Max discount_pct: {df_products['discount_pct'].max()}")
```

Новый метод — `.where(условие, замена)`. Читается как:
«оставь значение где условие True, иначе поставь NaN».

In [7]:
# Заменяем отрицательные скидки на NaN
df_products['discount_pct'] = df_products['discount_pct'].where(
    df_products['discount_pct'] >= 0, np.nan
)

# Проверяем
print(f"Отрицательных скидок осталось: {(df_products['discount_pct'] < 0).sum()}")
print(f"Min discount_pct теперь: {df_products['discount_pct'].min()}")
print(f"Max discount_pct: {df_products['discount_pct'].max()}")

Отрицательных скидок осталось: 0
Min discount_pct теперь: 0.0
Max discount_pct: 60.0


## Шаг 5: Исправление отрицательных скидок
```python
df_products['discount_pct'] = df_products['discount_pct'].where(
    df_products['discount_pct'] >= 0, np.nan
)

print(f"Отрицательных скидок осталось: {(df_products['discount_pct'] < 0).sum()}")
print(f"Min discount_pct теперь: {df_products['discount_pct'].min()}")
print(f"Max discount_pct: {df_products['discount_pct'].max()}")
```

**Результат:**
```
Отрицательных скидок осталось: 0
Min discount_pct теперь: 0.0
Max discount_pct: 60.0
```

Чисто. Диапазон `0–60%` — это экономически осмысленные значения.

---

### Метод .where() — важно понять

`.where(условие, замена)` работает **обратно** по отношению к тому как звучит:
```
оставь значение ГДЕ условие True
замени на NaN  ГДЕ условие False
```

Это контринтуитивно, но логично: «оставь там, где выполняется условие».
Альтернативный способ сделать то же самое через `.mask()` — он работает наоборот:
`mask(условие)` заменяет там где True. Обе команды полезны, выбирай что читается лучше.

---

### Итог двух шагов очистки

| Проблема | Было | Стало | Метод |
|---|---|---|---|
| `rating_count = -99` | 638 аномалий | NaN | `.replace(-99, np.nan)` |
| `discount_pct < 0` | 18 аномалий | NaN | `.where(>= 0, np.nan)` |

In [8]:
# Колонки которые удаляем и почему
cols_to_drop = [
    'color_name',          # 90% пропусков
    'badge_texts',         # 75% пропусков, маркетинговые теги
    'brand_name',          # всегда "adidas" — нулевая дисперсия
    'in_stock',            # 1 уникальное значение
    'market_record_source',# технический атрибут, 1 значение
    'size_record_source',  # технический атрибут
    'canonical_url',       # дублирует product_url
    'image_url',           # не нужен для ML
    'best_for_ids',        # сложный формат, мало пользы
    'seen_markets',        # дублирует seen_market_count
]

df_clean = df_products.drop(columns=cols_to_drop)

print(f"Колонок было:  {df_products.shape[1]}")
print(f"Колонок стало: {df_clean.shape[1]}")
print(f"\nОставшиеся колонки:")
print(df_clean.columns.tolist())

Колонок было:  35
Колонок стало: 25

Оставшиеся колонки:
['snapshot_date', 'country_code', 'product_name', 'model_number', 'currency', 'price_local', 'sale_price_local', 'gender_segment', 'size_label', 'category', 'subcategory', 'product_id', 'sku', 'base_model_number', 'size_count', 'size_labels', 'available_size_count', 'availability', 'availability_units', 'availability_status', 'discount_pct', 'rating', 'rating_count', 'product_url', 'seen_market_count']


## Шаг 6: Удаление бесполезных колонок
```python
cols_to_drop = [
    'color_name', 'badge_texts', 'brand_name', 'in_stock',
    'market_record_source', 'size_record_source', 'canonical_url',
    'image_url', 'best_for_ids', 'seen_markets',
]

df_clean = df_products.drop(columns=cols_to_drop)

print(f"Колонок было:  {df_products.shape[1]}")
print(f"Колонок стало: {df_clean.shape[1]}")
```

**Результат:** 35 → 25 колонок. Удалили 10 — почти треть.

---

### Метод .drop() и зачем создавать новый датафрейм

`.drop(columns=список)` — не меняет `df_products`, а **возвращает новую копию** без указанных колонок.
Поэтому мы сохраняем результат в `df_clean` — это финальный чистый датафрейм,
с которым будем работать дальше.

Принцип: всегда сохраняй оригинал (`df_products`) нетронутым.
Если что-то пошло не так — можно вернуться и пересобрать `df_clean` заново.

### Почему именно эти колонки

| Причина удаления | Колонки |
|---|---|
| Пропусков > 70% | `color_name`, `badge_texts` |
| Нулевая дисперсия (1 значение) | `brand_name`, `in_stock`, `market_record_source`, `size_record_source` |
| Дублируют другие колонки | `canonical_url` → `product_url`, `seen_markets` → `seen_market_count` |
| Нерелевантны для ML | `image_url`, `best_for_ids` |

In [9]:
# Проверяем пропуски в df_clean
missing = df_clean.isnull().sum() / len(df_clean) * 100
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.round(1))


rating_count            57.6
rating                  57.5
sale_price_local        31.9
discount_pct            17.9
subcategory             17.5
base_model_number        4.7
available_size_count     4.3
category                 2.1
sku                      1.3
availability_units       1.3
availability             1.0
availability_status      1.0
gender_segment           0.3
model_number             0.3
dtype: float64


## Шаг 7: Карта пропусков после очистки
```python
missing = df_clean.isnull().sum() / len(df_clean) * 100
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.round(1))
```

**Результат:** 14 колонок с пропусками. Делим их на три группы по стратегии.

---

### Стратегия заполнения для каждой группы

**Группа 1 — заполняем нулём** (NaN = «нет скидки» / «нет отзывов»):

| Колонка | Пропусков | Логика |
|---|---|---|
| `rating_count` | 57.6% | NaN = товар без отзывов → 0 |
| `rating` | 57.5% | NaN = нет рейтинга → 0 |
| `sale_price_local` | 31.9% | NaN = нет скидки → равно price_local |
| `discount_pct` | 17.9% | NaN = нет скидки → 0 |

**Группа 2 — заполняем модой** (наиболее частым значением):

| Колонка | Пропусков | Логика |
|---|---|---|
| `subcategory` | 17.5% | Текст — берём самое частое в категории |
| `category` | 2.1% | Мало пропусков — берём моду |
| `gender_segment` | 0.3% | Мало пропусков — берём моду |

**Группа 3 — оставляем как есть** (технические поля):

| Колонка | Пропусков | Логика |
|---|---|---|
| `base_model_number` | 4.7% | Технический ID — не используем в модели |
| `available_size_count` | 4.3% | Заполним медианой позже |
| `sku` | 1.3% | Идентификатор — не признак |
| `availability_units` | 1.3% | Технический атрибут |
| `availability` | 1.0% | Заполним модой |
| `availability_status` | 1.0% | Заполним модой |
| `model_number` | 0.3% | Идентификатор — не трогаем |

In [10]:
# Группа 1: заполняем нулём
df_clean['rating_count'] = df_clean['rating_count'].fillna(0)
df_clean['rating']       = df_clean['rating'].fillna(0)
df_clean['discount_pct'] = df_clean['discount_pct'].fillna(0)

# sale_price_local: NaN = нет скидки → цена равна обычной
df_clean['sale_price_local'] = df_clean['sale_price_local'].fillna(
    df_clean['price_local']
)

# Проверяем
cols = ['rating_count', 'rating', 'discount_pct', 'sale_price_local']
print(df_clean[cols].isnull().sum())

rating_count        0
rating              0
discount_pct        0
sale_price_local    0
dtype: int64
